# Horizon-sensitivity — Table V (h in {1, 10, 21}, embargo tracks h)

Mechanism check: the hybrid's edge over the frozen core should shrink as the forecast horizon h
grows toward VIX's own ~21-trading-day implied window. Targets for h != 5 are rebuilt directly from
`Close` (exactly mirroring the original data-builder's `build_target(r, h)`); h=5 is reported from
notebook 01 and folded in here for the compact table. The embargo depth always equals the run's own
h (verified by an assertion in `prepare_data`).

Approximate runtime: **~60 minutes** (3 horizons x 3 indices x 3 seeds = 27 trainings).


# Paper Experiments — Ablations & Baseline Correctors

The experiments that back the ICASSP paper's claims, all with the same frozen-core two-stage design,
multi-seed protocol, and Diebold–Mariano testing:

**Exp A (all 3 indices):** *what should the corrector be, and how should fusion happen?*
- corrector ∈ {**BI-Mamba**, **LSTM**, **GRU**, **MLP**} — parameter-matched, same everything else
- **output-fusion (ours, frozen core)** vs **input-fusion** (econometric predictors appended to the
  network inputs, à la Kim & Won 2018) with the same BI-Mamba corrector

**Exp B (IXIC):** *which parts of the design matter?*
- core composition: HAR only / HAR+IV / HAR+GARCH / full
- frozen vs jointly-trained core (both initialised at the OLS solution)
- unidirectional vs bidirectional scan (`n_scans` 1 vs 2)

All rows share identical test days (the row mask always requires the FULL core to exist, even when a
reduced core is used), so every pair is DM-comparable.

## 0 — Setup
Put `IXIC_vol.csv`, `NYA_vol.csv`, `DJI_vol.csv` (from `build_vol_dataset.py`) in the working directory.

In [1]:
!pip install -q torch numpy pandas arch scipy
import os
EXPECTED = ["IXIC_vol.csv", "NYA_vol.csv", "DJI_vol.csv"]
missing = [f for f in EXPECTED if not os.path.exists(f)]
print("found:", [f for f in EXPECTED if os.path.exists(f)])
if missing:
    print("MISSING:", missing, "\n-> upload them, or run build_vol_dataset.py first (needs internet):")
    print("   !pip install -q yfinance && python build_vol_dataset.py  (then move vol_dataset/*.csv here)")

found: ['IXIC_vol.csv', 'NYA_vol.csv', 'DJI_vol.csv']


In [2]:
import math, random, copy
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE, "| torch", torch.__version__)
if DEVICE == "cuda": print("gpu:", torch.cuda.get_device_name(0))

## 1 — Config
Every experiment is one `Config`. All ablation knobs below are implemented.

In [3]:
INDICES = ["DJI", "IXIC", "NYA"]        # all three are run; put <IDX>_vol.csv files next to the notebook

@dataclass
class Config:
    data_path: str = "DJI_vol.csv"                     # overridden per index by the runner
    target_col: str = "target_logrv"                    # log realized vol over the next `target_horizon` days
    not_features: List[str] = field(default_factory=lambda: ["Close", "target_logrv", "target_rv"])
    target_horizon: int = 5                             # h used to BUILD the target/baselines (must match build_vol_dataset.py)

    seq_in: int = 60                                    # long lookback: volatility has long memory
    horizon: int = 1                                    # model output dim (one target per sample)
    val_ratio: float = 0.15; test_ratio: float = 0.15

    d_model: int = 32; n_layer: int = 3; d_state: int = 128; expand: int = 2; d_conv: int = 3
    dt_rank: Optional[int] = None                       # None -> ceil(d_model/16)
    cheb_k: int = 3; embed: int = 10; hid: int = 32
    scan_chunk: int = 32                                # parallel-scan chunk (>= seq_len => single fully-parallel scan)

    batch_size: int = 32; epochs: int = 100; lr: float = 1e-3; weight_decay: float = 1e-4
    grad_clip: float = 5.0; patience: int = 15

    # ----- ablation knobs (pure BI-Mamba) -----
    scan_axis: str = "time"                             # "time" (baseline) | "feature" (variate-as-token)
    n_scans: int = 2                                    # 2 = bidirectional; >2 adds permuted scans
    use_registers: bool = False; n_registers: int = 4
    reg_control: bool = False                           # param-matched control for the register readout
    grad_checkpoint: bool = False                       # recompute scan in backward -> big GPU-memory saving on heavy rows
    hybrid: bool = False                                # add the HAR-X+VIX+GARCH core; BI-Mamba models the residual
    freeze_core: bool = True                            # lock the core to its OLS fit (two-stage) so you can't do worse than it
    corrector: str = "bimamba"                          # deep corrector backbone: bimamba | lstm | gru | mlp | patchtf | tsmixer (param-matched)
    input_fusion: bool = False                          # Kim&Won-style: core predictors appended to network INPUTS (use with hybrid=False)
    core_set: str = "full"                              # core composition: full | har | har_iv | har_garch
    qlike_align: bool = False                           # train & early-stop on QLIKE (variance loss) instead of MSE/RMSE
    cross_market: bool = False                          # input = features of ALL three indices (spillover); target stays this index
    cross_paths: List[str] = field(default_factory=lambda: ["DJI_vol.csv", "IXIC_vol.csv", "NYA_vol.csv"])

    seed: int = 1; device: str = DEVICE
    def __post_init__(self):
        if self.dt_rank is None: self.dt_rank = math.ceil(self.d_model / 16)

def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

def replace(cfg, **kw):
    d = asdict(cfg); d.update(kw); return Config(**d)

## 2 — Data pipeline (clean)

Read the per-index CSV, drop the non-feature columns (`Close`, `target_rv`), use `target_logrv` as
the label. Split chronologically, **standardize** features and target on **train only** (z-score is
right for log-RV and robust to the COVID spike that min-max would crush), then window. Each window's
label is `target_logrv` at its **last input day** — which already looks `h` days ahead, so the target
window never overlaps the inputs (leak-free by construction; the builder also self-checks this).

In [4]:
import warnings; warnings.filterwarnings("ignore")
from arch import arch_model

# ---- rolling GARCH(1,1) h-day vol forecast (causal: only past returns); cached so it runs once ----
def _garch_hday_logvol(r, h, refit=25, min_train=500):
    r = np.asarray(r, float); n = len(r); out = np.full(n, np.nan)
    mu = om = al = be = sbar = cv = None; last = -10**9
    for t in range(n):
        if t >= min_train and (t - last) >= refit:
            res = arch_model(pd.Series(r[:t+1]), mean="Constant", vol="Garch", p=1, q=1, dist="normal").fit(disp="off")
            pa = res.params; mu, om, al, be = pa["mu"], pa["omega"], pa["alpha[1]"], pa["beta[1]"]
            ps = al + be; sbar = om / (1 - ps) if 0 < ps < 1 else float(np.var(r[:t+1]))
            cv = float(np.asarray(res.conditional_volatility)[-1] ** 2); last = t
        if cv is None: continue
        s_next = om + al * (r[t] - mu) ** 2 + be * cv          # sigma^2_{t+1}
        ps = al + be; ksum = 0.0; ek = s_next
        for _ in range(h): ksum += ek; ek = sbar + ps * (ek - sbar)   # sum E[sigma^2_{t+1..t+h}]
        out[t] = 0.5 * np.log(ksum) - np.log(100)              # log h-day vol (decimal)
        cv = s_next
    return out

_GARCH_CACHE = {}
def garch_series(path, h, refit=25, min_train=500):
    key = (path, h, refit, min_train)
    if key not in _GARCH_CACHE:
        df = pd.read_csv(path, index_col=0, parse_dates=True).sort_index()
        r_full = (np.log(df["Close"].astype("float64")).diff() * 100).values      # NaN at [0]
        valid = np.isfinite(r_full)
        g = _garch_hday_logvol(r_full[valid], h, refit, min_train)                # GARCH on finite returns
        out = np.full(len(r_full), np.nan); out[valid] = g
        _GARCH_CACHE[key] = pd.Series(out, index=df.index)
    return _GARCH_CACHE[key]

class Standardizer:                                     # per-feature z-score (robust to outliers vs min-max)
    def fit(self, x):
        self.mu = x.mean(0); self.sd = x.std(0)
        self.sd = np.where(self.sd == 0, 1.0, self.sd); return self
    def transform(self, x): return (x - self.mu) / self.sd

class Standardizer1D:
    def fit(self, x):
        self.mu = float(np.mean(x)); self.sd = float(np.std(x)) or 1.0; return self
    def transform(self, x): return (x - self.mu) / self.sd
    def inverse(self, x):   return x * self.sd + self.mu

CORE_SETS = {"full": ["hd", "hw", "hm", "vix", "garch"], "har": ["hd", "hw", "hm"],
             "har_iv": ["hd", "hw", "hm", "vix"], "har_garch": ["hd", "hw", "hm", "garch"]}

def _core_frame(df, cfg):
    # ALWAYS builds all 5 columns; the mask uses the full frame so test rows are identical
    # across core_set / input_fusion choices. Only the selected subset feeds the core head / core OLS.
    close = df["Close"].astype("float64"); var = (np.log(close).diff()) ** 2; eps = 1e-12
    core = pd.DataFrame(index=df.index)
    core["hd"] = np.log(np.sqrt(var.rolling(1).sum()) + eps)            # HAR daily
    core["hw"] = np.log(np.sqrt(var.rolling(5).sum()) + eps)            # HAR weekly
    core["hm"] = np.log(np.sqrt(var.rolling(22).sum()) + eps)           # HAR monthly
    if "VIX" in df.columns: core["vix"] = np.log(df["VIX"].astype("float64") + eps)
    core["garch"] = garch_series(cfg.data_path, cfg.target_horizon).reindex(df.index)   # GARCH(1,1)
    return core

def _rebuild_target(df, h):
    # exactly mirrors build_vol_dataset.py's build_target(r, h): log realized vol over the
    # STRICTLY FUTURE window sum_{i=1..h} r_{t+i}^2 (shift(-h)); the CSV's target_logrv/target_rv
    # were built this way for h=5. The NaN tail (last h rows, no future window) is dropped later
    # by the existing feature/core mask, same as the builder's own usable-rows drop.
    r = np.log(df["Close"].astype("float64")).diff()
    var = r ** 2
    fut_sumsq = var.rolling(h).sum().shift(-h)
    rv = np.sqrt(fut_sumsq)
    return np.log(rv.replace(0, np.nan))

def _target_for_horizon(df, cfg):
    if cfg.target_horizon == 5:
        return df[cfg.target_col].astype("float64")            # CSV column, built for h=5
    return _rebuild_target(df, cfg.target_horizon)              # rebuilt from Close for any other h

def prepare_data(cfg):
    df = pd.read_csv(cfg.data_path, index_col=0, parse_dates=True).sort_index()   # chronological
    y = _target_for_horizon(df, cfg).astype("float32")                            # target_logrv (causal, future)
    feats = df.drop(columns=[c for c in cfg.not_features if c in df.columns]).astype("float32")
    if cfg.cross_market:                                                          # add the OTHER indices' full feature sets
        for p in [q for q in cfg.cross_paths if q != cfg.data_path]:
            od = pd.read_csv(p, index_col=0, parse_dates=True).sort_index()
            of = od.drop(columns=[c for c in cfg.not_features if c in od.columns]).astype("float32")
            of.columns = [f"{p.split('_')[0]}:{c}" for c in of.columns]
            feats = feats.join(of, how="inner")                                   # shared calendar (same builder)
        df, y = df.loc[feats.index], y.loc[feats.index]
    core_all = _core_frame(df, cfg)                                                # all 5 predictors (mask basis)

    m = feats.notna().all(1) & y.notna() & core_all.notna().all(1)                # mask on FULL core -> identical test rows
    core = core_all[CORE_SETS[cfg.core_set]]                                      # subset actually used by the core head
    if cfg.input_fusion:                                                          # Kim&Won-style: predictors as INPUTS
        feats = feats.join(core_all.astype("float32"), rsuffix="_core")
    feats, y, core = feats[m].values, y[m].values, core[m].values.astype("float32")
    N, Fdim = feats.shape; n_core = core.shape[1]

    n_test = int(N * cfg.test_ratio); n_val = int(N * cfg.val_ratio); n_train = N - n_val - n_test
    tr_end, val_end = n_train, n_train + n_val

    fscaler = Standardizer().fit(feats[:tr_end]); cscaler = Standardizer().fit(core[:tr_end])  # fit on TRAIN
    tscaler = Standardizer1D().fit(y[:tr_end])
    Xs = fscaler.transform(feats).astype("float32"); Cs = cscaler.transform(core).astype("float32")
    ys = tscaler.transform(y).astype("float32")

    # frozen-core OLS (standardized core -> standardized target), fit on TRAIN -> the hybrid's locked baseline
    A = np.c_[np.ones(tr_end), Cs[:tr_end]]; coef, *_ = np.linalg.lstsq(A, ys[:tr_end], rcond=None)
    core_coef = (coef[1:].astype("float32"), float(coef[0]))                       # (weights[n_core], bias)

    Xw, Cw, Yw, tgt = [], [], [], []
    for i in range(N - cfg.seq_in + 1):
        last = i + cfg.seq_in - 1                                                 # label/core at the LAST input day
        Xw.append(Xs[i:i+cfg.seq_in]); Cw.append(Cs[last]); Yw.append([ys[last]]); tgt.append(last)
    Xw = np.asarray(Xw, "float32"); Cw = np.asarray(Cw, "float32")
    Yw = np.asarray(Yw, "float32")[..., None]; tgt = np.asarray(tgt)

    h = cfg.target_horizon                                                        # embargo depth: forward target window
    def split(lo, hi):
        s = (tgt >= lo) & (tgt < hi)
        return s, torch.from_numpy(Xw[s]), torch.from_numpy(Cw[s]), torch.from_numpy(Yw[s])
    s_tr, Xtr, Ctr, Ytr = split(0, tr_end - h)          # purge last h train samples (target reaches into val)
    s_va, Xva, Cva, Yva = split(tr_end, val_end - h)    # purge last h val samples (target reaches into test)
    s_te, Xte, Cte, Yte = split(val_end, N)             # test needs no purge -- nothing follows it

    if s_tr.any() and s_va.any():
        assert tgt[s_tr].max() + h < tgt[s_va].min(), "train/val target-window overlap detected!"
    if s_va.any() and s_te.any():
        assert tgt[s_va].max() + h < tgt[s_te].min(), "val/test target-window overlap detected!"
    n_drop_tr = int(((tgt >= tr_end - h) & (tgt < tr_end)).sum())
    n_drop_va = int(((tgt >= val_end - h) & (tgt < val_end)).sum())
    print(f"purge[{cfg.data_path}, h={h}]: dropped {n_drop_tr} train-boundary + {n_drop_va} val-boundary rows | "
          f"train={int(s_tr.sum())} val={int(s_va.sum())} test={int(s_te.sum())} -- zero target-window overlap OK")

    return {"train": (Xtr, Ctr, Ytr), "val": (Xva, Cva, Yva), "test": (Xte, Cte, Yte)}, Fdim, tscaler, n_core, core_coef

def loaders(data, bs):
    return {k: torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, C, Y),
            batch_size=bs, shuffle=(k == "train"), drop_last=False) for k, (X, C, Y) in data.items()}

In [5]:
data, NF, ts, NCORE, CORE_COEF = prepare_data(Config())
for k, (X, C, Y) in data.items(): print(f"{k:5s} X={tuple(X.shape)}  core={tuple(C.shape)}  Y={tuple(Y.shape)}")
print("n_features:", NF, "| n_core (HAR-X+VIX+GARCH):", NCORE)

purge[DJI_vol.csv, h=5]: dropped 5 train-boundary + 5 val-boundary rows | train=2486 val=541 test=546 -- zero target-window overlap OK
train X=(2486, 60, 39)  core=(2486, 5)  Y=(2486, 1, 1)
val   X=(541, 60, 39)  core=(541, 5)  Y=(541, 1, 1)
test  X=(546, 60, 39)  core=(546, 5)  Y=(546, 1, 1)
n_features: 39 | n_core (HAR-X+VIX+GARCH): 5


## 3 — Model (pure BI-Mamba + optional hybrid core)

Backbone = **bidirectional Mamba** (chunked parallel scan) with the toggles built in; a mean-pool +
linear **readout** replaces the graph. `scan_axis="feature"` makes the features the sequence
(variate-as-token); `n_scans` sums that many permuted scan directions; `use_registers` inserts
evenly-spaced register tokens with a reduce-&-concat readout (`reg_control` = same params, no
registers). With `hybrid=True`, output = `linear(HAR_d, HAR_w, HAR_m, VIX) + BiMamba_correction`, and
the correction is zero-initialised so training starts exactly at the linear baseline.

In [6]:
from torch.utils.checkpoint import checkpoint

class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-5):
        super().__init__(); self.eps = eps; self.weight = nn.Parameter(torch.ones(d))
    def forward(self, x): return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps) * self.weight

class MambaBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.cfg = cfg; di = cfg.expand * cfg.d_model
        self.in_proj = nn.Linear(cfg.d_model, di * 2, bias=False)
        self.conv1d = nn.Conv1d(di, di, cfg.d_conv, groups=di, padding=cfg.d_conv - 1, bias=True)
        self.x_proj = nn.Linear(di, cfg.dt_rank + cfg.d_state * 2, bias=False)
        self.dt_proj = nn.Linear(cfg.dt_rank, di, bias=True)
        A = torch.arange(1, cfg.d_state + 1, dtype=torch.float32).repeat(di, 1)
        self.A_log = nn.Parameter(torch.log(A)); self.D = nn.Parameter(torch.ones(di))
        self.out_proj = nn.Linear(di, cfg.d_model, bias=False)
    def forward(self, x):
        b, l, _ = x.shape
        x_, res = self.in_proj(x).chunk(2, -1)
        x_ = F.silu(self.conv1d(x_.transpose(1, 2))[..., :l].transpose(1, 2))
        return self.out_proj(self._ssm(x_) * F.silu(res))
    def _ssm(self, x):
        A = -torch.exp(self.A_log.float()); D = self.D.float()
        delta, B, C = self.x_proj(x).split([self.cfg.dt_rank, self.cfg.d_state, self.cfg.d_state], -1)
        delta = F.softplus(self.dt_proj(delta))
        dA  = torch.exp(torch.einsum('bld,dn->bldn', delta, A))
        dBu = torch.einsum('bld,bln,bld->bldn', delta, B, x)
        return self._selective_scan(dA, dBu, C) + x * D
    def _selective_scan(self, dA, dBu, C):
        # chunked parallel scan: parallel within chunks, sequential across, n contracted per chunk
        L = dA.shape[1]; chunk = min(self.cfg.scan_chunk, L); carry, ys = None, []
        for s in range(0, L, chunk):
            a, bx, c = dA[:, s:s+chunk], dBu[:, s:s+chunk], C[:, s:s+chunk]
            h = self._chunk_scan(a, bx)
            if carry is not None: h = h + torch.cumprod(a, 1) * carry.unsqueeze(1)
            ys.append(torch.einsum('bcdn,bcn->bcd', h, c)); carry = h[:, -1]
        return torch.cat(ys, 1)
    @staticmethod
    def _chunk_scan(a, bx):                              # Hillis-Steele inclusive scan (zero init)
        L = a.shape[1]; shift = 1
        while shift < L:
            ap = F.pad(a,  (0, 0, 0, 0, shift, 0), value=1.0)[:, :L]
            bp = F.pad(bx, (0, 0, 0, 0, shift, 0), value=0.0)[:, :L]
            bx = a * bp + bx; a = a * ap; shift *= 2
        return bx

class ResidualBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__(); self.norm = nn.LayerNorm(cfg.d_model); self.mixer = MambaBlock(cfg)
    def forward(self, x): return self.mixer(self.norm(x))

class BiMamba(nn.Module):
    def __init__(self, cfg, nf):
        super().__init__(); self.cfg = cfg
        self.seq_len, self.token_in = (cfg.seq_in, nf) if cfg.scan_axis == "time" else (nf, cfg.seq_in)
        self.embedding = nn.Linear(self.token_in, cfg.d_model)
        self.ns = cfg.n_scans
        self.scans = nn.ModuleList([nn.ModuleList([ResidualBlock(cfg) for _ in range(cfg.n_layer)])
                                    for _ in range(self.ns)])
        self.use_reg = cfg.use_registers
        self.L_ext = self.seq_len + (cfg.n_registers if self.use_reg else 0)
        m = lambda nrm: nn.Sequential(nrm, nn.Linear(self.L_ext, cfg.hid), nn.ReLU(), nn.Linear(cfg.hid, self.L_ext))
        self.lin = nn.ModuleList([m(nn.LayerNorm(self.L_ext))] + [m(RMSNorm(self.L_ext)) for _ in range(cfg.n_layer - 1)])
        self.norm_f = nn.LayerNorm(cfg.d_model); self.head = nn.Linear(cfg.d_model, self.token_in)
        g = torch.Generator().manual_seed(cfg.seed)
        perms = [torch.arange(self.L_ext), torch.arange(self.L_ext).flip(0)]
        for _ in range(self.ns - 2): perms.append(torch.randperm(self.L_ext, generator=g))
        for i, p in enumerate(perms[:self.ns]):
            self.register_buffer(f"perm_{i}", p); self.register_buffer(f"inv_{i}", torch.argsort(p))
        self.global_dim = 0
        if self.use_reg:
            self.registers = nn.Parameter(torch.randn(cfg.n_registers, cfg.d_model) * 0.02)
            pos = torch.linspace(0, self.L_ext - 1, cfg.n_registers).round().long()
            assert len(torch.unique(pos)) == cfg.n_registers, "n_registers too large for seq_len"
            mask = torch.zeros(self.L_ext, dtype=torch.bool); mask[pos] = True
            self.register_buffer("reg_mask", mask)
            self.r = max(1, cfg.d_model // 4); self.reg_reduce = nn.Linear(cfg.d_model, self.r)
            self.global_dim = cfg.n_registers * self.r
        elif cfg.reg_control:
            self.r = max(1, cfg.d_model // 4); self.reg_reduce = nn.Linear(cfg.d_model, self.r)
            self.global_dim = cfg.n_registers * self.r
    def _insert(self, x):
        b, S, d = x.shape
        out = torch.empty(b, self.L_ext, d, device=x.device, dtype=x.dtype)
        out[:, self.reg_mask] = self.registers.to(x.dtype).expand(b, -1, -1)
        out[:, ~self.reg_mask] = x
        return out
    def _block(self, blk, x):
        if self.cfg.grad_checkpoint and self.training:
            return checkpoint(blk, x, use_reentrant=False)
        return blk(x)
    def forward(self, x):                               # x: [b, L, F]
        if self.cfg.scan_axis == "feature": x = x.transpose(1, 2)
        x = self.embedding(x)
        if self.use_reg: x = self._insert(x)
        for i in range(self.cfg.n_layer):
            agg = x
            for s in range(self.ns):
                p, inv = getattr(self, f"perm_{s}"), getattr(self, f"inv_{s}")
                agg = agg + self._block(self.scans[s][i], x[:, p])[:, inv]
            x = agg
            x = self.lin[i](x.transpose(1, 2)).transpose(1, 2) + x
        x = self.norm_f(x)
        gvec = None
        if self.use_reg:
            gvec = self.reg_reduce(x[:, self.reg_mask]).flatten(1)
            x = x[:, ~self.reg_mask]
        elif self.cfg.reg_control:
            gvec = self.reg_reduce(x).mean(1).repeat(1, self.cfg.n_registers)
        out = self.head(x)
        if self.cfg.scan_axis == "feature": out = out.transpose(1, 2)
        return out, gvec                                # [b, L, F], global vec or None

class SeqRNN(nn.Module):
    # bidirectional LSTM/GRU corrector with the same interface as BiMamba: forward -> ([b,seq,nf], None)
    global_dim = 0
    def __init__(self, kind, nf, hidden):
        super().__init__()
        RNN = nn.LSTM if kind == "lstm" else nn.GRU
        self.rnn = RNN(nf, hidden, num_layers=1, bidirectional=True, batch_first=True)
        self.proj = nn.Linear(2 * hidden, nf)
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.proj(out), None

class WindowMLP(nn.Module):
    # MLP over the flattened window (no temporal structure) -> [b,1,nf]; the temporal-inductive-bias control
    global_dim = 0
    def __init__(self, nf, seq_in, hidden):
        super().__init__()
        self.net = nn.Sequential(nn.Flatten(), nn.Linear(seq_in * nf, hidden), nn.GELU(), nn.Linear(hidden, nf))
    def forward(self, x):
        return self.net(x).unsqueeze(1), None

class PatchTF(nn.Module):
    # PatchTST-style corrector: non-overlapping patches (patch_len=stride=10, seq_in=60 -> 6 patches),
    # each patch flattened and linearly projected to d_model, learnable positional embedding, a
    # standard 2-layer/4-head TransformerEncoder, mean-pool over patches -> linear head back to nf.
    global_dim = 0
    def __init__(self, nf, seq_in, d_model, patch_len=10, n_head=4):
        super().__init__()
        assert seq_in % patch_len == 0
        self.patch_len = patch_len; self.n_patches = seq_in // patch_len
        d_model = max(n_head, (d_model // n_head) * n_head)          # must be divisible by n_head
        self.proj_in = nn.Linear(patch_len * nf, d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, self.n_patches, d_model))
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_head, dim_feedforward=2 * d_model,
                                            activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.head = nn.Linear(d_model, nf)
    def forward(self, x):                                            # x: [b, seq_in, nf]
        b, L, F = x.shape
        p = x.reshape(b, self.n_patches, self.patch_len * F)
        h = self.proj_in(p) + self.pos_emb                            # [b, n_patches, d_model]
        h = self.encoder(h)
        return self.head(h.mean(1)).unsqueeze(1), None                # [b, 1, nf], None

class _MixerBlock(nn.Module):
    def __init__(self, seq_in, nf, hidden):
        super().__init__()
        self.norm1 = nn.LayerNorm(nf); self.norm2 = nn.LayerNorm(nf)
        self.time_mlp = nn.Sequential(nn.Linear(seq_in, hidden), nn.GELU(), nn.Linear(hidden, seq_in))
        self.feat_mlp = nn.Sequential(nn.Linear(nf, hidden), nn.GELU(), nn.Linear(hidden, nf))
    def forward(self, x):                                            # x: [b, L, F]
        x = x + self.time_mlp(self.norm1(x).transpose(1, 2)).transpose(1, 2)   # time-mixing, shared over F
        x = x + self.feat_mlp(self.norm2(x))                                    # feature-mixing, shared over L
        return x

class TSMixer(nn.Module):
    # TSMixer-style corrector: 2 blocks of (time-mixing MLP, feature-mixing MLP), residual + LayerNorm
    # each, mean-pool over time -> same [b,1,nf] interface as the other correctors.
    global_dim = 0
    def __init__(self, nf, seq_in, hidden, n_blocks=2):
        super().__init__()
        self.blocks = nn.ModuleList([_MixerBlock(seq_in, nf, hidden) for _ in range(n_blocks)])
    def forward(self, x):
        for blk in self.blocks: x = blk(x)
        return x.mean(1, keepdim=True), None                          # [b, 1, nf], None

class _sLSTMCell(nn.Module):
    # sLSTM-style scalar-memory recurrent cell (xLSTM paper, Beck et al. 2024): exponential
    # input/forget gates stabilized via a running log-space max (m_t), normalizer state n_t.
    # NOTE: this is the sLSTM (scalar-memory) variant only -- no matrix-memory mLSTM -- a
    # deliberate, documented simplification (a full mLSTM would be disproportionate here).
    def __init__(self, d_in, d_hidden):
        super().__init__()
        self.d_hidden = d_hidden
        self.Wz = nn.Linear(d_in, d_hidden); self.Rz = nn.Linear(d_hidden, d_hidden, bias=False)
        self.Wi = nn.Linear(d_in, d_hidden); self.Ri = nn.Linear(d_hidden, d_hidden, bias=False)
        self.Wf = nn.Linear(d_in, d_hidden); self.Rf = nn.Linear(d_hidden, d_hidden, bias=False)
        self.Wo = nn.Linear(d_in, d_hidden); self.Ro = nn.Linear(d_hidden, d_hidden, bias=False)
    def forward(self, x):                                             # x: [b, L, d_in]
        b, L, _ = x.shape
        h = x.new_zeros(b, self.d_hidden); c = x.new_zeros(b, self.d_hidden)
        n = x.new_ones(b, self.d_hidden); m = x.new_zeros(b, self.d_hidden)
        outs = []
        for t in range(L):
            xt = x[:, t]
            zt = torch.tanh(self.Wz(xt) + self.Rz(h))
            i_tilde = self.Wi(xt) + self.Ri(h)
            f_tilde = self.Wf(xt) + self.Rf(h)
            o_t = torch.sigmoid(self.Wo(xt) + self.Ro(h))
            m_new = torch.maximum(f_tilde + m, i_tilde)               # log-space stabilizer
            i_t = torch.exp(i_tilde - m_new); f_t = torch.exp(f_tilde + m - m_new)
            c = f_t * c + i_t * zt
            n = f_t * n + i_t
            h = o_t * (c / (n + 1e-6))
            m = m_new
            outs.append(h)
        return torch.stack(outs, 1)                                   # [b, L, d_hidden]

class _XLSTMBlock(nn.Module):
    def __init__(self, d_model, expand=2):
        super().__init__()
        self.norm = nn.LayerNorm(d_model); d_inner = d_model * expand
        self.up = nn.Linear(d_model, d_inner)
        self.cell = _sLSTMCell(d_inner, d_inner)
        self.down = nn.Linear(d_inner, d_model)
    def forward(self, x):
        u = self.up(self.norm(x))
        u = self.cell(u)
        return x + self.down(u)

class XLSTM(nn.Module):
    # xLSTM-style corrector: sLSTM blocks (scalar memory, exponential gating) wrapped in the usual
    # pre-LayerNorm + residual + up/down-projection block, 2 blocks.
    global_dim = 0
    def __init__(self, nf, seq_in, d_model, n_blocks=2):
        super().__init__()
        self.inp = nn.Linear(nf, d_model)
        self.blocks = nn.ModuleList([_XLSTMBlock(d_model) for _ in range(n_blocks)])
        self.head = nn.Linear(d_model, nf)
    def forward(self, x):
        u = self.inp(x)
        for blk in self.blocks: u = blk(u)
        return self.head(u), None                                     # [b, L, nf], None

class ITransformer(nn.Module):
    # iTransformer-style corrector: each of the F feature-series (length L) is embedded into ONE
    # token via a linear layer; a standard 2-layer/4-head TransformerEncoder attends ACROSS the F
    # variate-tokens (not across time -- the deliberate "inverted" design); each token is then
    # projected back to a length-L series and transposed back to [b, L, nf].
    global_dim = 0
    def __init__(self, nf, seq_in, d_model, n_head=4):
        super().__init__()
        d_model = max(n_head, (d_model // n_head) * n_head)
        self.embed = nn.Linear(seq_in, d_model)
        layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=n_head, dim_feedforward=2 * d_model,
                                            activation="gelu", batch_first=True)
        self.encoder = nn.TransformerEncoder(layer, num_layers=2)
        self.project = nn.Linear(d_model, seq_in)
    def forward(self, x):                                             # x: [b, L, nf]
        xt = x.transpose(1, 2)                                        # [b, nf, L] -- nf tokens
        h = self.encoder(self.embed(xt))                              # attention across variates
        return self.project(h).transpose(1, 2), None                  # [b, L, nf], None

def _n_params(m): return sum(p.numel() for p in m.parameters())

def make_backbone(cfg, nf):
    # parameter-match every alternative corrector to the BI-Mamba backbone (binary search on hidden size)
    if cfg.corrector == "bimamba": return BiMamba(cfg, nf)
    target = _n_params(BiMamba(cfg, nf))
    build = {"lstm": lambda h: SeqRNN("lstm", nf, h), "gru": lambda h: SeqRNN("gru", nf, h),
             "mlp": lambda h: WindowMLP(nf, cfg.seq_in, h),
             "patchtf": lambda h: PatchTF(nf, cfg.seq_in, h),
             "tsmixer": lambda h: TSMixer(nf, cfg.seq_in, h),
             "xlstm": lambda h: XLSTM(nf, cfg.seq_in, h),
             "itransformer": lambda h: ITransformer(nf, cfg.seq_in, h)}[cfg.corrector]
    hi_map = {"lstm": 768, "gru": 768, "patchtf": 512, "itransformer": 512}
    lo, hi = 4, hi_map.get(cfg.corrector, 4096)
    while lo < hi:
        mid = (lo + hi) // 2
        if _n_params(build(mid)) < target: lo = mid + 1
        else: hi = mid
    h = min([max(lo - 1, 4), lo], key=lambda k: abs(_n_params(build(k)) - target))
    return build(h)

class BiMambaNet(nn.Module):
    # Pure bidirectional Mamba (graph removed). Optional HAR-X+VIX+GARCH core => hybrid.
    def __init__(self, cfg, nf, n_core=0, core_coef=None):
        super().__init__(); self.cfg = cfg; self.nf = nf
        self.backbone = make_backbone(cfg, nf)                       # BI-Mamba or a param-matched corrector
        self.readout = nn.Linear(nf, cfg.horizon)                    # deep correction from pooled output
        self.hybrid = bool(cfg.hybrid and n_core > 0)
        if self.hybrid:
            self.core = nn.Linear(n_core, cfg.horizon)               # linear HAR-X+VIX+GARCH baseline
            nn.init.zeros_(self.readout.weight); nn.init.zeros_(self.readout.bias)   # start AT the baseline
            if core_coef is not None:                                # ALWAYS initialise the core at its OLS fit
                w, b = core_coef
                with torch.no_grad():
                    self.core.weight.copy_(torch.as_tensor(w, dtype=torch.float32).view(1, -1))
                    self.core.bias.fill_(float(b))
            if cfg.freeze_core:                                      # two-stage: lock it (else: joint fine-tuning)
                self.core.weight.requires_grad_(False); self.core.bias.requires_grad_(False)
        if self.backbone.global_dim:
            self.reg_head = nn.Linear(self.backbone.global_dim, cfg.horizon)
            if self.hybrid: nn.init.zeros_(self.reg_head.weight); nn.init.zeros_(self.reg_head.bias)
    def forward(self, x, core=None):
        h, gvec = self.backbone(x)                                   # h: [b, seq_in, nf]
        out = self.readout(h.mean(1))                               # [b, horizon]
        if gvec is not None: out = out + self.reg_head(gvec)
        if self.hybrid and core is not None: out = out + self.core(core)
        return out.unsqueeze(-1)                                     # [b, horizon, 1]

print("baseline params:", sum(p.numel() for p in BiMambaNet(Config(), NF, NCORE, CORE_COEF).parameters()))

baseline params: 203251


## 4 — Metrics (volatility)
On **log realized vol**: **RMSE**, **MAE** (↓); **R²** (Mincer–Zarnowitz, ↑); **Corr** (Pearson, ↑). On **variance** (`σ²=exp(2·logRV)`): **QLIKE** (↓), the standard robust volatility loss. Early stopping uses validation RMSE.

In [7]:
def _pearson(a, b):
    a, b = a - a.mean(), b - b.mean()
    den = a.std(unbiased=False) * b.std(unbiased=False)
    return torch.tensor(float("nan")) if den == 0 else (a * b).mean() / den

def _vol_metrics(pred, true):                              # pred/true = log realized vol
    err = pred - true
    rmse = (err ** 2).mean().sqrt().item(); mae = err.abs().mean().item()
    ss_res = (err ** 2).sum(); ss_tot = ((true - true.mean()) ** 2).sum()
    r2 = (1 - ss_res / ss_tot).item() if ss_tot > 0 else float("nan")
    vp, vt = torch.exp(2 * pred), torch.exp(2 * true)      # variance
    qlike = (vt / vp - torch.log(vt / vp) - 1).mean().item()
    return {"RMSE": rmse, "MAE": mae, "QLIKE": qlike, "R2": r2, "Corr": _pearson(pred, true).item()}

@torch.no_grad()
def evaluate(model, loader, tscaler, device, return_preds=False):
    model.eval(); ps, tsx = [], []
    for X, C, Y in loader:
        ps.append(model(X.to(device), C.to(device)).cpu().reshape(-1)); tsx.append(Y.reshape(-1))
    pred = torch.from_numpy(tscaler.inverse(torch.cat(ps).numpy()))   # predicted log-RV
    true = torch.from_numpy(tscaler.inverse(torch.cat(tsx).numpy()))  # actual log-RV
    m = _vol_metrics(pred, true)
    return (m, pred.numpy(), true.numpy()) if return_preds else m

## 5 — Train · multi-seed · ablation harness

In [8]:
def qlike_loss_std(pred_z, true_z, sd):        # differentiable QLIKE in standardized log-RV space
    r = torch.clamp((2.0 * sd) * (true_z - pred_z), -10.0, 10.0)
    return (torch.exp(r) - r - 1).mean()

def train_once(cfg, data, nf, tscaler, n_core, core_coef, verbose=False):
    set_seed(cfg.seed); dl = loaders(data, cfg.batch_size)
    model = BiMambaNet(cfg, nf, n_core, core_coef).to(cfg.device)
    opt = torch.optim.Adam(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sd = float(tscaler.sd); _mse = nn.MSELoss()
    lossf = (lambda p, y: qlike_loss_std(p, y, sd)) if cfg.qlike_align else (lambda p, y: _mse(p, y))
    es = "QLIKE" if cfg.qlike_align else "RMSE"        # early-stop metric aligned with the loss
    best, best_state, wait = float("inf"), None, 0
    for ep in range(cfg.epochs):
        model.train()
        for X, Cc, Y in dl["train"]:
            X, Cc, Y = X.to(cfg.device), Cc.to(cfg.device), Y.to(cfg.device)
            opt.zero_grad(); loss = lossf(model(X, Cc), Y); loss.backward()
            if cfg.grad_clip: nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            opt.step()
        vm = evaluate(model, dl["val"], tscaler, cfg.device)[es]
        if vm < best: best, best_state, wait = vm, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if cfg.patience and wait >= cfg.patience: break
        if verbose and ep % 10 == 0: print(f"  epoch {ep:3d} | val {es} {vm:.4f}")
    if best_state: model.load_state_dict(best_state)
    return evaluate(model, dl["test"], tscaler, cfg.device, return_preds=True)

def run_multiseed(cfg, seeds=(1, 2, 3), verbose=False):
    data, nf, ts, n_core, core_coef = prepare_data(cfg)
    mets, preds, true = [], [], None
    for s in seeds:
        m, p, t = train_once(replace(cfg, seed=s), data, nf, ts, n_core, core_coef, verbose)
        mets.append(m); preds.append(p); true = t
    ens = np.mean(np.stack(preds), 0)                  # seed-ensemble prediction (used for DM)
    stats = {k: (float(np.mean([r[k] for r in mets])), float(np.std([r[k] for r in mets]))) for k in mets[0]}
    return stats, ens, true

def vol_baselines(cfg, return_preds=False):
    import numpy.linalg as la
    df = pd.read_csv(cfg.data_path, index_col=0, parse_dates=True).sort_index()
    feats = df.drop(columns=[c for c in cfg.not_features if c in df.columns])
    y = _target_for_horizon(df, cfg)
    close = df["Close"].astype("float64"); var = (np.log(close).diff()) ** 2
    h = cfg.target_horizon; eps = 1e-12
    persist = np.log(np.sqrt(var.rolling(h).sum()) + eps)
    ew = var.ewm(alpha=1 - 0.94, adjust=False).mean()
    ewma = 0.5 * np.log(ew * h + eps)
    hd = np.log(np.sqrt(var.rolling(1).sum()) + eps)
    hw = np.log(np.sqrt(var.rolling(5).sum()) + eps)
    hm = np.log(np.sqrt(var.rolling(22).sum()) + eps)
    garch = garch_series(cfg.data_path, h).reindex(df.index)
    cols = {"y": y, "persist": persist, "ewma": ewma, "hd": hd, "hw": hw, "hm": hm, "garch": garch}
    if "VIX" in feats.columns: cols["vix"] = np.log(feats["VIX"].astype("float64") + eps)
    sub = pd.DataFrame(cols, index=df.index)
    full_cols = [c for c in CORE_SETS["full"] if c in sub.columns]
    core_cols = [c for c in CORE_SETS[cfg.core_set] if c in sub.columns]
    sub = sub[feats.notna().all(1) & y.notna() & sub[full_cols].notna().all(1)]   # mask on FULL core = same rows as the model

    N = len(sub); n_test = int(N * cfg.test_ratio); n_val = int(N * cfg.val_ratio); n_train = N - n_val - n_test
    tr = sub.iloc[:n_train].dropna(); te = sub.iloc[n_train + n_val:]
    tt = torch.tensor(te["y"].values)
    def ols(cols_):
        A = np.c_[np.ones(len(tr)), tr[cols_].values]; b, *_ = la.lstsq(A, tr["y"].values, rcond=None)
        return np.c_[np.ones(len(te)), te[cols_].values] @ b
    P = {"persistence (RW)": te["persist"].values, "EWMA (l=.94)": te["ewma"].values,
         "GARCH(1,1)": te["garch"].values, "HAR-RV": ols(["hd", "hw", "hm"])}
    if "vix" in sub.columns:
        P["VIX-only"] = ols(["vix"])
        P[f"core OLS ({cfg.core_set})"] = ols(core_cols)
    out = {n: _vol_metrics(torch.tensor(np.asarray(p)), tt) for n, p in P.items()}
    if return_preds: return out, {n: np.asarray(p, float) for n, p in P.items()}, te["y"].values.astype(float)
    return out

PREDS, TRUES = {}, {}                                   # tag -> {method: test predictions (log-RV)}, tag -> true
def run_ablation(named, seeds=(1, 2, 3), baselines=True, tag=None):
    res, preds, true_m = {}, {}, None
    for name, cfg in named.items():
        print(f"[run] {name}"); res[name], p, true_m = run_multiseed(cfg, seeds); preds[name] = p
    df = pd.DataFrame({n: {k: f"{m:.4f} ± {s:.4f}" for k, (m, s) in r.items()} for n, r in res.items()}).T
    if baselines:
        b, bp, true_b = vol_baselines(list(named.values())[0], return_preds=True)
        bdf = pd.DataFrame({n: {k: f"{v:.4f}" for k, v in mt.items()} for n, mt in b.items()}).T
        df = pd.concat([bdf, df])
        preds = {**bp, **preds}
        if true_m is not None and len(true_m) != len(true_b):
            print(f"!! test-set mismatch: model {len(true_m)} rows vs baselines {len(true_b)} (cross_market calendar?) — DM will skip unequal pairs")
        true_m = true_b if true_m is None else true_m
    if tag is not None: PREDS[tag], TRUES[tag] = preds, np.asarray(true_m, float)
    return df


## h=5 target-rebuild verification and h=1 numerical-stability check

In [9]:
# ---- (c) h=5 target-rebuild verification: must match the CSV column to numerical precision ----
print("h=5 target-rebuild verification (rebuilt from Close vs. CSV target_logrv column):")
for idx in INDICES:
    df = pd.read_csv(f"{idx}_vol.csv", index_col=0, parse_dates=True).sort_index()
    y_csv = df["target_logrv"].astype("float64")
    y_rebuilt = _rebuild_target(df, 5)
    diff = (y_csv - y_rebuilt).abs()
    print(f"  {idx}: max abs diff = {diff.max():.3e}  (n valid pairs: {int(diff.notna().sum())})")

# ---- h=1 numerical-stability check: does the codebase's existing eps=1e-12 floor matter here? ----
print("\nh=1 numerical-stability check (persistence (RW) baseline, IXIC, as a representative probe):")
_cfg_probe = replace(Config(), data_path="IXIC_vol.csv", target_horizon=1)
_b, _bp, _true_b = vol_baselines(_cfg_probe, return_preds=True)
_pred, _true = _bp["persistence (RW)"], _true_b
_vp, _vt = np.exp(2 * _pred), np.exp(2 * _true)
_raw_qlike = float(np.mean(_vt / _vp - np.log(_vt / _vp) - 1))
_eps = 1e-12
_vp_floored = np.maximum(_vp, _eps)
_floored_qlike = float(np.mean(_vt / _vp_floored - np.log(_vt / _vp_floored) - 1))
_n_floor_triggers = int((_vp < _eps).sum())
print(f"  raw QLIKE = {_raw_qlike:.2f}  |  eps-floored (eps={_eps:.0e}) QLIKE = {_floored_qlike:.2f}  "
      f"|  floor triggered on {_n_floor_triggers}/{len(_vp)} days")
if abs(_raw_qlike - _floored_qlike) < 1e-9:
    print("  -> the standard eps=1e-12 floor (same value used throughout this codebase, e.g. _core_frame/"
          "vol_baselines log(.)+eps terms) makes NO difference here: predicted variances at h=1 are already "
          "many orders of magnitude above 1e-12. NOT applying an ad-hoc larger/different epsilon (that would "
          "be inventing a new, unspecified threshold, i.e. silent clipping) -- the h=1 QLIKE inflation is a "
          "genuine property of a few extreme forecast-error days at daily granularity (a quiet day followed by "
          "a volatility spike, or vice versa), not a numerical floor artifact. Reporting as-is; see per-index "
          "tables below, where this shows up as an elevated h=1 QLIKE across methods (worst for persistence).")
else:
    print("  -> eps floor changed the result; using the floored value going forward (reported explicitly).")


h=5 target-rebuild verification (rebuilt from Close vs. CSV target_logrv column):
  DJI: max abs diff = 8.882e-16  (n valid pairs: 4144)
  IXIC: max abs diff = 8.882e-16  (n valid pairs: 4144)
  NYA: max abs diff = 8.882e-16  (n valid pairs: 4144)

h=1 numerical-stability check (persistence (RW) baseline, IXIC, as a representative probe):


  raw QLIKE = 1844.55  |  eps-floored (eps=1e-12) QLIKE = 1844.55  |  floor triggered on 0/546 days
  -> the standard eps=1e-12 floor (same value used throughout this codebase, e.g. _core_frame/vol_baselines log(.)+eps terms) makes NO difference here: predicted variances at h=1 are already many orders of magnitude above 1e-12. NOT applying an ad-hoc larger/different epsilon (that would be inventing a new, unspecified threshold, i.e. silent clipping) -- the h=1 QLIKE inflation is a genuine property of a few extreme forecast-error days at daily granularity (a quiet day followed by a volatility spike, or vice versa), not a numerical floor artifact. Reporting as-is; see per-index tables below, where this shows up as an elevated h=1 QLIKE across methods (worst for persistence).


## Horizon sweep, per-horizon DM, and compact tables

In [10]:
QUICK_TEST = False
EPOCHS = 3 if QUICK_TEST else 100
SEEDS  = (1,) if QUICK_TEST else (1, 2, 3)
PAT    = 0 if QUICK_TEST else 15
print(f"QUICK_TEST={QUICK_TEST} -> epochs={EPOCHS}, seeds={SEEDS}, patience={PAT}, device={DEVICE}")

HORIZONS = [1, 10, 21]
tables_hz_purged = {}
for h in HORIZONS:
    for idx in INDICES:
        path = f"{idx}_vol.csv"
        print("\n" + "=" * 70 + f"\n### PURGED h={h} — {idx}\n" + "=" * 70)
        tables_hz_purged[(h, idx)] = run_ablation({
            "Hybrid (BI-Mamba)": replace(Config(), data_path=path, hybrid=True, weight_decay=1e-3,
                                         target_horizon=h, epochs=EPOCHS, patience=PAT),
        }, seeds=SEEDS, tag=f"PurgedH{h}-{idx}")
        display(tables_hz_purged[(h, idx)])


QUICK_TEST=False -> epochs=100, seeds=(1, 2, 3), patience=15, device=cuda

### PURGED h=1 — DJI
[run] Hybrid (BI-Mamba)


purge[DJI_vol.csv, h=1]: dropped 1 train-boundary + 1 val-boundary rows | train=2488 val=545 test=546 -- zero target-window overlap OK


,RMSE,MAE,QLIKE,R2,Corr
persistence (RW),1.6290,1.2020,565.0337,-0.9276,0.0363
EWMA (l=.94),1.3926,0.9911,1.6127,-0.4088,0.0956
"GARCH(1,1)",1.3863,0.9862,1.5578,-0.3961,0.1380
HAR-RV,1.1744,0.8858,4.1174,-0.0018,0.1255
VIX-only,1.1419,0.8556,2.9890,0.0529,0.2363
core OLS (full),1.1492,0.8646,3.3080,0.0406,0.2181
Hybrid (BI-Mamba),1.1609 ± 0.0116,0.8541 ± 0.0044,2.7560 ± 0.1585,0.0210 ± 0.0196,0.2112 ± 0.0122



### PURGED h=1 — IXIC
[run] Hybrid (BI-Mamba)
purge[IXIC_vol.csv, h=1]: dropped 1 train-boundary + 1 val-boundary rows | train=2490 val=545 test=546 -- zero target-window overlap OK


,RMSE,MAE,QLIKE,R2,Corr
persistence (RW),1.6535,1.2741,1844.5539,-1.0017,-0.0013
EWMA (l=.94),1.3927,1.0111,1.6521,-0.4200,0.1246
"GARCH(1,1)",1.3682,0.9862,1.6040,-0.3706,0.1784
HAR-RV,1.1606,0.9156,4.2071,0.0138,0.1571
VIX-only,1.1435,0.9029,4.3839,0.0427,0.2238
core OLS (full),1.1391,0.8965,4.1950,0.0500,0.2392
Hybrid (BI-Mamba),1.1361 ± 0.0002,0.8787 ± 0.0057,3.4335 ± 0.2479,0.0550 ± 0.0003,0.2455 ± 0.0035


...

purge[IXIC_vol.csv, h=21]: dropped 21 train-boundary + 21 val-boundary rows | train=2460 val=522 test=543 -- zero target-window overlap OK


,RMSE,MAE,QLIKE,R2,Corr
persistence (RW),0.4128,0.3451,0.3892,-0.2971,0.3504
EWMA (l=.94),0.4116,0.3369,0.3583,-0.2899,0.2622
"GARCH(1,1)",0.3482,0.2856,0.2679,0.0771,0.4091
HAR-RV,0.3525,0.2759,0.3799,0.0539,0.4023
VIX-only,0.3661,0.2653,0.4751,-0.0203,0.3956
core OLS (full),0.3590,0.2626,0.4462,0.0191,0.4176
Hybrid (BI-Mamba),0.3422 ± 0.0027,0.2553 ± 0.0058,0.3469 ± 0.0182,0.1085 ± 0.0143,0.3609 ± 0.0077



### PURGED h=21 — NYA
[run] Hybrid (BI-Mamba)


purge[NYA_vol.csv, h=21]: dropped 21 train-boundary + 21 val-boundary rows | train=2456 val=522 test=543 -- zero target-window overlap OK


,RMSE,MAE,QLIKE,R2,Corr
persistence (RW),0.3878,0.2923,0.3694,-0.3739,0.3158
EWMA (l=.94),0.3916,0.3016,0.3667,-0.4005,0.2492
"GARCH(1,1)",0.3727,0.3008,0.2792,-0.2686,0.3556
HAR-RV,0.3190,0.2391,0.3054,0.0705,0.3321
VIX-only,0.3190,0.2437,0.2632,0.0704,0.4402
core OLS (full),0.3153,0.2377,0.2662,0.0917,0.4409
Hybrid (BI-Mamba),0.3141 ± 0.0049,0.2327 ± 0.0087,0.2844 ± 0.0135,0.0986 ± 0.0284,0.4281 ± 0.0036


### DM per (index, horizon) — Hybrid vs core OLS (full) / Hybrid vs VIX-only

Same HAC(h-1)+HLN machinery, daily QLIKE (and SE), seed-ensemble predictions; DM lag = h-1 for each
horizon (h=1 -> plain DM+HLN with lag 0).


In [11]:
from scipy import stats as _st

def _daily_qlike(pred, true): r = 2 * (true - pred); return np.exp(r) - r - 1
def _daily_se(pred, true):    return (pred - true) ** 2

def dm_test(lossA, lossB, h):
    d = np.asarray(lossA, float) - np.asarray(lossB, float)
    n = len(d); dbar = d.mean(); dc = d - dbar
    g = [float(np.dot(dc[k:], dc[:n - k])) / n for k in range(max(h - 1, 0) + 1)]
    var = (g[0] + 2 * sum(g[1:])) / n
    if var <= 0: return float("nan"), float("nan")
    dm = dbar / np.sqrt(var) * np.sqrt((n + 1 - 2 * h + h * (h - 1) / n) / n)   # HLN
    return dm, 2 * _st.t.sf(abs(dm), df=n - 1)

HZ_QLIKE_PURGED = {}
hz_dm_rows_purged = []
for h in HORIZONS:
    for idx in INDICES:
        P, true = PREDS[f"PurgedH{h}-{idx}"], TRUES[f"PurgedH{h}-{idx}"]
        HZ_QLIKE_PURGED[(h, idx)] = {name: _vol_metrics(torch.tensor(P[name]), torch.tensor(true))["QLIKE"]
                                      for name in ("VIX-only", "core OLS (full)", "Hybrid (BI-Mamba)") if name in P}
        for a, b in [("Hybrid (BI-Mamba)", "core OLS (full)"), ("Hybrid (BI-Mamba)", "VIX-only")]:
            if a not in P or b not in P or len(P[a]) != len(true) or len(P[b]) != len(true):
                hz_dm_rows_purged.append({"h": h, "index": idx, "A": a, "B": b, "note": "skipped"}); continue
            qa, qb = _daily_qlike(P[a], true), _daily_qlike(P[b], true)
            dm_q, p_q = dm_test(qa, qb, h)
            dm_s, p_s = dm_test(_daily_se(P[a], true), _daily_se(P[b], true), h)
            verdict = ("A better" if dm_q < 0 else "B better") if (p_q == p_q and p_q < 0.05) else "≈ (n.s.)"
            hz_dm_rows_purged.append({"h": h, "index": idx, "A": a, "B": b,
                                "QLIKE A": round(float(qa.mean()), 4), "QLIKE B": round(float(qb.mean()), 4),
                                "DM(QLIKE)": round(dm_q, 2), "p(QLIKE)": round(p_q, 4),
                                "DM(SE)": round(dm_s, 2), "p(SE)": round(p_s, 4), "verdict": verdict})

hz_dm_table_purged = pd.DataFrame(hz_dm_rows_purged)
display(hz_dm_table_purged)


,h,index,A,B,QLIKE A,QLIKE B,DM(QLIKE),p(QLIKE),DM(SE),p(SE),verdict
0,1,DJI,Hybrid (BI-Mamba),core OLS (full),2.6867,3.3080,-8.56,0.0000,1.26,0.2081,A better
1,1,DJI,Hybrid (BI-Mamba),VIX-only,2.6867,2.9890,-4.27,0.0000,1.95,0.0517,A better
2,1,IXIC,Hybrid (BI-Mamba),core OLS (full),3.4178,4.1950,-9.83,0.0000,-0.86,0.3896,A better
3,1,IXIC,Hybrid (BI-Mamba),VIX-only,3.4178,4.3839,-8.63,0.0000,-1.23,0.2199,A better
4,1,NYA,Hybrid (BI-Mamba),core OLS (full),2.2118,2.6235,-8.11,0.0000,2.42,0.0157,A better
5,1,NYA,Hybrid (BI-Mamba),VIX-only,2.2118,2.3299,-2.06,0.0395,1.01,0.3125,A better
6,10,DJI,Hybrid (BI-Mamba),core OLS (full),0.3167,0.3851,-1.02,0.3070,1.30,0.1930,≈ (n.s.)
7,10,DJI,Hybrid (BI-Mamba),VIX-only,0.3167,0.3486,-0.70,0.4870,2.04,0.0422,≈ (n.s.)
8,10,IXIC,Hybrid (BI-Mamba),core OLS (full),0.3894,0.5300,-2.40,0.0167,-0.31,0.7570,A better
9,10,IXIC,Hybrid (BI-Mamba),VIX-only,0.3894,0.5721,-2.29,0.0222,-0.66,0.5092,A better


In [12]:
print("(h=5 is the main Table I/II/IV run -- see that section/results/table_I,II above; not "
      "reproduced here since this notebook is standalone and does not carry that run's PREDS.)\n")

p_lookup_purged = {(r["h"], r["index"], r["A"], r["B"]): r for r in hz_dm_rows_purged}
for idx in INDICES:
    rows = {}
    for h in HORIZONS:
        q = HZ_QLIKE_PURGED[(h, idx)]
        p_core = p_lookup_purged.get((h, idx, "Hybrid (BI-Mamba)", "core OLS (full)"), {}).get("p(QLIKE)")
        p_vix  = p_lookup_purged.get((h, idx, "Hybrid (BI-Mamba)", "VIX-only"), {}).get("p(QLIKE)")
        rows[f"h={h}"] = {"VIX-only": round(q.get("VIX-only", float("nan")), 4),
                           "core OLS (full)": round(q.get("core OLS (full)", float("nan")), 4),
                           "Hybrid (BI-Mamba)": round(q.get("Hybrid (BI-Mamba)", float("nan")), 4),
                           "p(Hybrid vs core)": p_core, "p(Hybrid vs VIX)": p_vix}
    print(f"--- {idx} (purged) ---")
    display(pd.DataFrame(rows).T)
    edges = [HZ_QLIKE_PURGED[(h, idx)]["core OLS (full)"] - HZ_QLIKE_PURGED[(h, idx)]["Hybrid (BI-Mamba)"] for h in HORIZONS]
    shrinks = all(edges[i] <= edges[i - 1] for i in range(1, len(edges)))
    tag = "SHRINKS monotonically toward h=21" if shrinks else "NOT monotonic"
    print(f"{idx}: Hybrid-core edge at h=1,10,21 = {[round(e, 4) for e in edges]} -> {tag}")
    print()


(h=5 is the main Table I/II/IV run -- see that section/results/table_I,II above; not reproduced here since this notebook is standalone and does not carry that run's PREDS.)

--- DJI (purged) ---


,VIX-only,core OLS (full),Hybrid (BI-Mamba),p(Hybrid vs core),p(Hybrid vs VIX)
h=1,2.9890,3.3080,2.6867,0.0000,0.0000
h=10,0.3486,0.3851,0.3167,0.3070,0.4870
h=21,0.2521,0.2604,0.2626,0.9561,0.7701


DJI: Hybrid-core edge at h=1,10,21 = [0.6213, 0.0684, -0.0022] -> SHRINKS monotonically toward h=21

--- IXIC (purged) ---


,VIX-only,core OLS (full),Hybrid (BI-Mamba),p(Hybrid vs core),p(Hybrid vs VIX)
h=1,4.3839,4.1950,3.4178,0.0000,0.0000
h=10,0.5721,0.5300,0.3894,0.0167,0.0222
h=21,0.4751,0.4462,0.3438,0.0905,0.0831


IXIC: Hybrid-core edge at h=1,10,21 = [0.7771, 0.1406, 0.1024] -> SHRINKS monotonically toward h=21

--- NYA (purged) ---


,VIX-only,core OLS (full),Hybrid (BI-Mamba),p(Hybrid vs core),p(Hybrid vs VIX)
h=1,2.3299,2.6235,2.2118,0.0000,0.0395
h=10,0.3425,0.3727,0.3582,0.4640,0.2838
h=21,0.2632,0.2662,0.2819,0.4862,0.5581


NYA: Hybrid-core edge at h=1,10,21 = [0.4117, 0.0145, -0.0157] -> SHRINKS monotonically toward h=21

